# FREUID smoke run (IDE -> Colab GPU remote kernel)

Open this notebook **in your IDE**, connect its kernel to a remote Colab server, then run it.
Compute runs on the Colab GPU; output shows in your IDE.

---

## STEP 1 — on Colab (browser; bring up the tunnel)

New Colab notebook -> Runtime type **GPU (T4)** -> add 🔑 Secret `NGROK_AUTH_TOKEN`.
Paste the cells below into Colab and run them in order.

```python
# [Colab 1] check GPU + install + clone
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -> set Runtime to GPU')
!pip install -q pyngrok
!git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
%cd kaggle_freuid_2026
!pip install -q -e .
```

```python
# [Colab 2] write the smoke config
%%writefile configs/smoke.yaml
name: smoke
seed: 42
data_dir: data_smoke
image_size: 224
val_fraction: 0.1
backbone: tf_efficientnetv2_s.in21k
pretrained: true
epochs: 2
batch_size: 16
lr: 0.0003
weight_decay: 0.0001
num_workers: 2
limit: 100
```

```python
# [Colab 3] upload + unzip data (pick data_smoke_100.zip)
from google.colab import files
files.upload()
!unzip -q -o data_smoke_100.zip
!ls data_smoke/train/train | wc -l
```

```python
# [Colab 4] GPU kernel + ngrok tunnel -> print URL
import subprocess, time, socket
from google.colab import userdata
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN').strip()
JUPYTER_TOKEN = 'freuid'
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
ngrok.kill()
log = open('/tmp/jl.log', 'w')
subprocess.Popen(['jupyter','server','--ip=0.0.0.0','--port=8888','--no-browser','--allow-root',
    f'--ServerApp.token={JUPYTER_TOKEN}','--ServerApp.password=','--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True','--ServerApp.allow_remote_access=True',
    '--ServerApp.root_dir=/content/kaggle_freuid_2026'], stdout=log, stderr=subprocess.STDOUT)
for i in range(30):
    time.sleep(1)
    s = socket.socket()
    if s.connect_ex(('127.0.0.1', 8888)) == 0: s.close(); break
    s.close()
print(ngrok.connect(8888, 'http').public_url + '/?token=' + JUPYTER_TOKEN)
```

-> Copy the printed `https://....ngrok-free.app/?token=freuid`. (Keep this Colab tab open.)

---

## STEP 2 — in my IDE (this notebook)
1. Top-right **kernel picker** -> `Select Another Kernel...` -> `Existing Jupyter Server...` -> paste the URL -> `Python 3`
2. Run the cells below

In [ ]:
# Connection check: are we attached to the Colab GPU?
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('cwd:', os.getcwd())

In [ ]:
# Run the smoke training (executes on the Colab GPU)
%cd /content/kaggle_freuid_2026
!python -m freuid.train --config configs/smoke.yaml